# Diabetes Risk Prediction — Feature Engineering & Baseline Model

**Goal:** Prepare the data for modeling and train an interpretable baseline model.

Sections:
1. Load cleaned data
2. Train/test split
3. Feature engineering (encoding, scaling)
4. Handle class imbalance
5. Baseline model: Logistic Regression
6. Evaluation (precision, recall, F1, ROC-AUC)
7. Key takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

## 1. Load Cleaned Data

Re-load the data and re-apply the cleaning steps from notebook 01 (drop duplicates).
If you saved a cleaned CSV from notebook 01, load that instead.

In [ ]:
from ucimlrepo import fetch_ucirepo

cdc_diabetes = fetch_ucirepo(id=891)
X_raw = cdc_diabetes.data.features
y_raw = cdc_diabetes.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df = df.drop_duplicates().reset_index(drop=True)

target_col = 'Diabetes_binary'
print(df.shape)
df.head()

## 2. Train/Test Split

Split BEFORE any scaling or resampling to avoid data leakage. Use stratified splitting
so both sets preserve the class imbalance ratio.

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('\nTrain class balance:')
print(y_train.value_counts(normalize=True))
print('\nTest class balance:')
print(y_test.value_counts(normalize=True))

## 3. Feature Engineering

Most features in this dataset are already binary (0/1) or ordinal, which is convenient.
The main things to handle:
- Scale continuous features (BMI, MentHlth, PhysHlth) for logistic regression
- Optionally create derived features (e.g. BMI categories)

Fit the scaler on TRAIN only, then apply to both train and test — never fit on test data.

In [ ]:
# Identify continuous features that benefit from scaling
continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
continuous_features = [c for c in continuous_features if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

X_train_scaled.head()

In [ ]:
# Optional: derived feature example — BMI category
def bmi_category(bmi):
    if bmi < 18.5:
        return 'underweight'
    elif bmi < 25:
        return 'normal'
    elif bmi < 30:
        return 'overweight'
    else:
        return 'obese'

# Example only — decide if you want to add this as a feature or keep BMI continuous
# X_train['BMI_category'] = X_train['BMI'].apply(bmi_category)
# pd.get_dummies(X_train['BMI_category'], prefix='bmi')

## 4. Handle Class Imbalance

This dataset is imbalanced. Two common approaches:
- **Class weighting** (simpler, no synthetic data): pass `class_weight='balanced'` to the model
- **Resampling** (SMOTE, undersampling): rebalances the training data directly

We'll start with class weighting for the baseline — it's simpler and a fair first attempt.

In [ ]:
# Check the imbalance ratio
ratio = y_train.value_counts(normalize=True)
print(f"Class imbalance ratio (train): {ratio[0]:.1%} negative vs {ratio[1]:.1%} positive")

## 5. Baseline Model: Logistic Regression

In [ ]:
baseline_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

baseline_model.fit(X_train_scaled, y_train)

y_pred = baseline_model.predict(X_test_scaled)
y_pred_proba = baseline_model.predict_proba(X_test_scaled)[:, 1]

## 6. Evaluation

Remember: accuracy is misleading here. Focus on precision, recall, F1, and ROC-AUC.

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes/Pre']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'ROC-AUC: {roc_auc:.3f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Diabetes', 'Diabetes/Pre'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Baseline Logistic Regression')
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Baseline Model')
plt.legend()
plt.show()

In [ ]:
# Which features matter most in this baseline model?
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': baseline_model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.head(10))

plt.figure(figsize=(8, 6))
sns.barplot(data=coef_df.head(10), x='coefficient', y='feature')
plt.title('Top 10 Features by Logistic Regression Coefficient (Absolute Value)')
plt.show()

## 7. Key Takeaways

*(Fill this in — a few plain-English sentences on baseline performance and what it means.
e.g. 'The baseline model achieves an ROC-AUC of X, correctly flagging Y% of at-risk patients
(recall) at the cost of Z% false positives. The strongest predictors were...')*

- 
- 
- 

**Next notebook:** `03_model_comparison_and_evaluation.ipynb` — try Random Forest and XGBoost,
compare against this baseline, and tune for better recall on the positive class.